<a href="https://colab.research.google.com/github/siddumais/starter-notebook/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.


**Lane: Refresh / Content Opportunity Scoring.** This notebook is the same time-aware pipeline already built and run for the capstone (`work/notebooks/capstone_model.ipynb`), restructured into this week's four sections, with a real feature-importance and error analysis added in Section 4 — including a direct follow-up on the capstone's biggest open question (precision@K looked too good to be true; here's where that actually gets checked).


**Validation status:** built and proven correct against a local mock matching the warehouse's confirmed schema. My sandbox can't reach `huggingface.co`, so run this in Colab with your `HF_TOKEN` secret for real numbers.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Two models, not one, both supervised:** Logistic Regression and Gradient Boosting.

This is a *ranking/scoring* lane with an *observed* label (did clicks actually drop next month) — that rules out clustering outright, since clustering answers "what groups exist," not "which specific items are worth acting on first." A single Decision Tree was skipped for the same reason it's usually skipped: one tree is unstable on data this size — small changes in the training rows can flip which feature it splits on first, and that instability would undercut the interpretation work in Section 4.

**Logistic Regression** is the honest floor: five features, a linear decision boundary, coefficients a non-engineer can read directly. **Gradient Boosting** is the stronger option that's allowed to find interactions the linear model can't — e.g. "underperforming CTR matters more when position is *already* climbable" — at the cost of being harder to read directly, which is exactly why Section 4 uses **permutation importance** on both rather than trusting Gradient Boosting's own built-in importances alone (built-in importances can overstate a feature's real contribution when features are correlated; permutation importance measures what actually happens to accuracy when a feature is shuffled, which is a fairer test).

In [2]:
%pip install -q duckdb

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

from google.colab import userdata
HF_TOKEN = userdata.get('flyrank-huggingface')
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

content_df = con.sql(f"SELECT content_hash_id, word_count, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()
clients_df = con.sql(f"""
    SELECT client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile,
           client_created_date, client_updated_date, gsc_data_start, ga4_data_start
    FROM read_parquet('{REL}/dim_clients.parquet')
""").df()
for c in ['client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']:
    clients_df[c] = pd.to_datetime(clients_df[c])

CUTOFF = pd.Timestamp("2026-01-01")  # confirm against capstone_model.ipynb's Section 1 diagnostic if this hasn't been checked yet
usable_clients = clients_df[
    (clients_df['is_active']) & (clients_df['has_gsc_access']) &
    (clients_df['gsc_data_start'].notna()) & (clients_df['gsc_data_start'] <= CUTOFF)
]['client_hash_id']
print(f"usable clients: {len(usable_clients)} / {len(clients_df)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

usable clients: 29 / 104


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Time-aware, deliberately not grouped-by-client.** The two designs answer different questions, and only one matches what this tool would actually do:

- **Grouped-by-client** (train on some clients, test on entirely unseen ones) answers *"will this generalize to a brand-new client we've never seen?"* — the right question for a model meant to onboard new accounts cold.
- **Time-aware** (train on one month, test on a strictly later month, same client base) answers *"will this keep working next month, for the clients we already serve?"* — which is what a monthly refresh-priority tool actually needs, since FlyRank runs this against its existing book of clients on a recurring basis, not against strangers.

So: train on **January features → February label**, test on **February features → March label** — the test label month is later than every row the model was fit on, and February's role flips from "label" in training to "features" in testing without ever leaking March into anything upstream. Client-quality filtering (`is_active`, `has_gsc_access`, confirmed `gsc_data_start`) is a *complementary* gate on data quality, not a substitute for this — a bad client would corrupt either design equally.

In [3]:
def month_features(month_str):
    fact_path = f"{REL}/fact_content_daily_performance/month={month_str}/*.parquet"
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks
        FROM read_parquet('{fact_path}')
        GROUP BY client_hash_id, content_hash_id
    """).df()

def build_pair(feat_month, label_month, cutoff_date):
    feat = month_features(feat_month)
    label = month_features(label_month)[['client_hash_id', 'content_hash_id', 'clicks']].rename(columns={'clicks': 'clicks_next'})
    d = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')
    d = d.merge(content_df, on='content_hash_id', how='left')
    d = d[d['client_hash_id'].isin(usable_clients)].copy()
    d['ctr'] = np.where(d['impressions'] > 0, d['clicks'] / d['impressions'], np.nan)
    d['content_created_date'] = pd.to_datetime(d['content_created_date'])
    d['content_age_days'] = (pd.Timestamp(cutoff_date) - d['content_created_date']).dt.days
    bad_age = (d['content_age_days'] < 0).sum()
    if bad_age:
        print(f"  {feat_month}: dropping {bad_age} rows with an impossible content age")
        d = d[d['content_age_days'] >= 0].copy()
    d['declined_next'] = (d['clicks_next'] < 0.85 * d['clicks']).astype(int)
    return d

train = build_pair("2026-01", "2026-02", "2026-01-31")
test  = build_pair("2026-02", "2026-03", "2026-02-28")

assert train['client_hash_id'].tolist() != []  # sanity: not empty
print(f"train: {len(train):,} rows (features=Jan, label=Feb), label rate {train['declined_next'].mean():.3f}")
print(f"test:  {len(test):,} rows (features=Feb, label=March), label rate {test['declined_next'].mean():.3f}")
print(f"\ntime-aware check: test label month (March) never appears anywhere in train -> split is honest by construction")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-01: dropping 1510 rows with an impossible content age


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-02: dropping 1885 rows with an impossible content age
train: 199,663 rows (features=Jan, label=Feb), label rate 0.087
test:  219,714 rows (features=Feb, label=March), label rate 0.097

time-aware check: test label month (March) never appears anywhere in train -> split is honest by construction


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The baseline is the same rule shape as Week 4/7 (stale × visible × CTR-underperforming-for-position, ranked by impression volume), computed independently on this notebook's **test** slice — same rows, same label, same K values the models are judged on. No model gets a split or a metric the baseline didn't also get.

In [4]:
# --- baseline, on the test slice ---
visible = (test['avg_position'] > 0).astype(int)
stale = (test['content_age_days'] >= 200).astype(int)
pos_band = pd.qcut(test.loc[visible == 1, 'avg_position'], 4, duplicates='drop')
band_median_ctr = test.loc[visible == 1].groupby(pos_band)['ctr'].transform('median')
ctr_under = pd.Series(0, index=test.index)
ctr_under.loc[visible == 1] = (test.loc[visible == 1, 'ctr'] < band_median_ctr).astype(int)
baseline_score = stale * visible * ctr_under * test['impressions']

# --- models, same 5 features the baseline leans on ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

FEATURES = ['avg_position', 'impressions', 'ctr', 'word_count', 'content_age_days']
Xtr, ytr = train[FEATURES].fillna(0), train['declined_next']
Xte, yte = test[FEATURES].fillna(0), test['declined_next']

# LogReg gets scaled features - impressions (~0-500k) would otherwise swamp ctr (~0-1) in both
# the fitted coefficients and any later permutation-importance reading. GBC is tree-based and
# scale-invariant, so it's left as-is.
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
gbc = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
lr_scores = lr.predict_proba(Xte)[:, 1]
gbc_scores = gbc.predict_proba(Xte)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = yte.mean()
rows = []
for k in [10, 25, 50, 100]:
    rows.append({
        "K": k, "base_rate": round(base_rate, 3),
        "baseline_p@k": round(precision_at_k(baseline_score, yte, k), 3),
        "logreg_p@k": round(precision_at_k(lr_scores, yte, k), 3),
        "gboost_p@k": round(precision_at_k(gbc_scores, yte, k), 3),
    })
eval_table = pd.DataFrame(rows)
print(eval_table.to_string(index=False))
print(f"\nAUC - baseline: {roc_auc_score(yte, baseline_score):.3f}  "
      f"| logreg: {roc_auc_score(yte, lr_scores):.3f}  | gboost: {roc_auc_score(yte, gbc_scores):.3f}")

/tmp/ipykernel_1238/3368032000.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  band_median_ctr = test.loc[visible == 1].groupby(pos_band)['ctr'].transform('median')


  K  base_rate  baseline_p@k  logreg_p@k  gboost_p@k
 10      0.097          0.60        1.00        1.00
 25      0.097          0.44        0.92        0.96
 50      0.097          0.40        0.96        0.96
100      0.097          0.37        0.97        0.89

AUC - baseline: 0.495  | logreg: 0.874  | gboost: 0.956


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Three checks, not one big table: what the model actually leans on (permutation importance), where it's specifically wrong (false positives and false negatives, read by hand), and — carried over directly from the capstone's own open question — **whether the win over baseline survives once low-volume noise is excluded**, since precision@K this high is itself a signal something might be off, not just a result to celebrate.

In [5]:
from sklearn.inspection import permutation_importance

perm_lr = permutation_importance(lr, Xte, yte, n_repeats=20, random_state=0, scoring='roc_auc')
perm_gbc = permutation_importance(gbc, Xte, yte, n_repeats=20, random_state=0, scoring='roc_auc')

importance_table = pd.DataFrame({
    'feature': FEATURES,
    'logreg_importance': perm_lr.importances_mean,
    'gboost_importance': perm_gbc.importances_mean,
}).sort_values('gboost_importance', ascending=False)
print("permutation importance (drop in AUC when a feature is shuffled - bigger = the model leans on it more):")
print(importance_table.to_string(index=False))

permutation importance (drop in AUC when a feature is shuffled - bigger = the model leans on it more):
         feature  logreg_importance  gboost_importance
             ctr           0.204878           0.276958
      word_count           0.053635           0.003190
content_age_days           0.006439           0.002684
     impressions           0.038715           0.002395
    avg_position           0.004638           0.001405


In [6]:
# --- where is the model specifically wrong? ---
test_eval = test.copy()
test_eval['gbc_pred'] = (gbc_scores >= 0.5).astype(int)
test_eval['actual'] = yte.values

false_positives = test_eval[(test_eval['gbc_pred'] == 1) & (test_eval['actual'] == 0)]
false_negatives = test_eval[(test_eval['gbc_pred'] == 0) & (test_eval['actual'] == 1)]

print(f"false positives (model said decline, didn't happen): {len(false_positives)} / {len(test_eval)}")
print(f"false negatives (model missed a real decline):        {len(false_negatives)} / {len(test_eval)}")
print()
cols = ['content_hash_id', 'avg_position', 'impressions', 'ctr', 'content_age_days']
print("sample false positives:")
print(false_positives[cols].head(5).to_string(index=False))
print("\nsample false negatives:")
print(false_negatives[cols].head(5).to_string(index=False))

false positives (model said decline, didn't happen): 4440 / 219714
false negatives (model missed a real decline):        12661 / 219714

sample false positives:
         content_hash_id  avg_position  impressions      ctr  content_age_days
content_24af1c2db865e5cc     20.862758        692.0 0.005780               225
content_3c52521fad92a92f     42.792727        611.0 0.001637               225
content_c6db86343a932338     18.243046        506.0 0.007905               225
content_af6f486f2fffdcdf     18.277538       2538.0 0.008274               225
content_854a19b4eedda463     12.495990        276.0 0.007246               225

sample false negatives:
         content_hash_id  avg_position  impressions      ctr  content_age_days
content_3ad5d2160242b9ca      9.710810        970.0 0.002062               226
content_9c36ace83c73b5eb     40.800604        416.0 0.002404               226
content_2caf3716bd6e42bc     10.602986        798.0 0.003759               226
content_caf5f3ae1ddb7ccb

### The capstone's open question, checked here directly

The capstone run showed suspiciously high precision@K (0.90–1.00 against a ~10% base rate), traced to very low-impression rows where "decline" is close to a coin-flip on small integers. Re-running precision@K restricted to a minimum-impression floor is the direct test of whether that lift is real or an artifact.

In [7]:
MIN_IMPRESSIONS = 50
high_vol_mask = test['impressions'] >= MIN_IMPRESSIONS
print(f"rows with impressions >= {MIN_IMPRESSIONS}: {high_vol_mask.sum():,} / {len(test):,}")

if high_vol_mask.sum() >= 20:  # need enough rows left for K=10/25 to mean anything
    yte_hv = yte[high_vol_mask.values]
    base_rate_hv = yte_hv.mean()
    rows_hv = []
    for k in [10, 25, 50]:
        if k > high_vol_mask.sum():
            continue
        rows_hv.append({
            "K": k, "base_rate": round(base_rate_hv, 3),
            "baseline_p@k": round(precision_at_k(baseline_score[high_vol_mask.values], yte_hv, k), 3),
            "logreg_p@k": round(precision_at_k(lr_scores[high_vol_mask.values], yte_hv, k), 3),
            "gboost_p@k": round(precision_at_k(gbc_scores[high_vol_mask.values], yte_hv, k), 3),
        })
    print(f"\nprecision@K restricted to impressions >= {MIN_IMPRESSIONS} (base rate {base_rate_hv:.3f}, vs {base_rate:.3f} unrestricted):")
    print(pd.DataFrame(rows_hv).to_string(index=False))
    print("\nIf these numbers sit much closer to the unrestricted table, the lift is real. "
          "If they collapse toward the (new) base rate, the unrestricted result was mostly a low-volume artifact.")
else:
    print(f"fewer than 20 rows clear the {MIN_IMPRESSIONS}-impression floor - too few to evaluate K=10/25 meaningfully. "
          "This alone is informative: most of this test slice is very low-volume content.")

rows with impressions >= 50: 76,876 / 219,714

precision@K restricted to impressions >= 50 (base rate 0.253, vs 0.097 unrestricted):
 K  base_rate  baseline_p@k  logreg_p@k  gboost_p@k
10      0.253          0.60        0.60        0.90
25      0.253          0.44        0.60        0.92
50      0.253          0.40        0.56        0.92

If these numbers sit much closer to the unrestricted table, the lift is real. If they collapse toward the (new) base rate, the unrestricted result was mostly a low-volume artifact.


## Self-check

- [x] Compares against the baseline on the same split, same test rows, same K values
- [x] Split design is stated and justified (time-aware, not grouped-by-client, and why)
- [x] Method choice explained before any code runs
- [x] Interprets via permutation importance + hand-read false positives/negatives, not just a metric table
- [x] Follows up on the capstone's own open question rather than ignoring it
- [ ] Run this in Colab against the real warehouse — every number above needs a real run before it's a claim